In [1]:
import os
import re
import json
import sys
import time
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from dotenv import load_dotenv
from mnt.skills.agente_mysql.helpers import MySQLAgent
import logging

load_dotenv()

# Logger do Maestro — troque para logging.DEBUG para ver detalhes de execução
logging.basicConfig(
    level=logging.DEBUG, # WARNING,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s"
)
logger = logging.getLogger("maestro")

sys.path.insert(0, os.path.abspath("."))

In [2]:
# Caminho base das skills (pode ser absoluto ou relativo ao notebook)
SKILLS_DIR = "mnt/skills"
SKILL_FILE = "SKILL.md"

def _parse_frontmatter(content: str) -> Tuple[Dict, str]:
    """Extrai frontmatter YAML (entre ---) e retorna (metadados, resto do conteúdo)."""
    meta = {}
    body = content
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)", content, re.DOTALL)
    if match:
        fm, body = match.group(1), match.group(2)
        for line in fm.split("\n"):
            if ":" in line:
                key, val = line.split(":", 1)
                meta[key.strip()] = val.strip().strip("'\"").replace("\n", " ").strip()
    return meta, body.strip()

def load_skills(base_dir: str) -> List[Dict]:
    """
    Carrega todas as skills a partir de base_dir.
    Cada subpasta contendo SKILL.md vira uma skill.
    Retorna lista de dicts: id, name, description, content (texto completo).
    """
    skills = []
    base = os.path.abspath(base_dir)
    if not os.path.isdir(base):
        return skills
    for entry in sorted(os.listdir(base)):
        path = os.path.join(base, entry)
        skill_file = os.path.join(path, SKILL_FILE)
        if os.path.isdir(path) and os.path.isfile(skill_file):
            with open(skill_file, "r", encoding="utf-8") as f:
                raw = f.read()
            meta, body = _parse_frontmatter(raw)
            skill_id = entry
            name = meta.get("name", skill_id)
            description = meta.get("description", "")
            skills.append({
                "id": skill_id,
                "name": name,
                "description": description,
                "model": meta.get("model"),   # modelo definido no SKILL.md
                "content": raw,
                "body": body,
            })
    return skills

def get_skill_by_id(skills: List[Dict], skill_id: str) -> Optional[Dict]:
    """Retorna a skill com o id dado ou None."""
    for s in skills:
        if s["id"] == skill_id:
            return s
    return None

# Carrega todas as skills
skills_loaded = load_skills(SKILLS_DIR)
print(f"Skills carregadas: {len(skills_loaded)}")
for s in skills_loaded:
    print(f"  - {s['id']}: {s.get('name', s['id'])}")
    # print(s['content'])

Skills carregadas: 13
  - agente_analise_os: agente_analise_os
  - agente_analise_queries_os: agente_analise_queries_os
  - agente_cientifico: agente_cientifico
  - agente_dados: agente_dados
  - agente_financeiro: agente_financeiro
  - agente_juridico: agente_juridico
  - agente_mercado: agente_mercado
  - agente_mysql: agente_mysql
  - agente_negocios: agente_negocios
  - agente_saude: agente_saude
  - agente_tecnico: agente_tecnico
  - avaliador_coerencia: avaliador_coerencia
  - maestro: maestro


In [6]:
agente_analise_os = get_skill_by_id(skills_loaded, 'agente_analise_os')
# print(agente_analise_os['content'])

In [7]:
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = os.environ.get("MODELO_DEFAULT")

In [ ]:
modelo = agente_analise_os.get("model") or model
user_message = "quem é você"
resp = client.chat.completions.create(
    model=modelo,
    messages=[
        {"role": "system", "content": agente_analise_os["content"]},
        {"role": "user", "content": user_message},
    ],
)
print(resp.choices[0].message.content)